# MIMIC-IV V5.5 Official DiCE Extension Paper

No-retraining extension of the accepted Q1 V5.5 plaintext paper run. This
notebook runs the **official `dice-ml==0.12` Random and Genetic methods** on
the frozen outer-test cohort. It reuses the checksum-locked split,
preprocessor and MLP checkpoint; neither the MLP nor any GAN is trained.

The full run evaluates 100 factuals per direction. `timeout`, `no_cf`, and
implementation failures stay in the denominator. Outcome/constraint metrics
are directly comparable with V5.5. Candidate-budget cost is not claimed to be
matched because official DiCE exposes native sample/iteration controls rather
than the proposed method's counted population budget.


In [ ]:
import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "scikit-learn==1.6.1", "dice-ml==0.12"
])


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, time, zipfile
import joblib
import numpy as np
import pandas as pd
import torch

DATASET = 'mimic'
RUN_MODE = 'paper'
PROTOCOL = "Q1_V55_OFFICIAL_DICE_EXTENSION_" + RUN_MODE.upper()
PER_DIRECTION = 100
OPERATING_BUDGET = 512
K = 10
MARGIN = 0.10
BASELINE_SEED = 11
TIMEOUT_SECONDS = 20
RANDOM_SAMPLE_SIZE = 10_000
GENETIC_MAXITERATIONS = 300
ROBUSTNESS_REPETITIONS = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT = Path("/kaggle/working") / f"{DATASET}_official_dice_extension_{RUN_MODE}"
OUTPUT.mkdir(parents=True, exist_ok=True)

print({
    "protocol": PROTOCOL, "dataset": DATASET, "device": DEVICE,
    "queries_per_direction": PER_DIRECTION, "K": K, "margin": MARGIN,
    "baseline_seed": BASELINE_SEED, "timeout_seconds": TIMEOUT_SECONDS,
    "random_sample_size": RANDOM_SAMPLE_SIZE,
    "genetic_maxiterations": GENETIC_MAXITERATIONS,
    "retraining": False,
})


## Frozen-source audit and exact reconstruction

The notebook rejects an unaccepted source or checksum mismatch. The raw data
are only transformed with the already fitted preprocessor so that
DomainProjector, constraints, plausibility references, and the test row
indices are identical to the accepted V5.5 run.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def discover_source():
    roots = []
    for acceptance in Path("/kaggle/input").rglob("v55_paper_acceptance.json"):
        root = acceptance.parent
        if (root / "paper_training_handoff/classifier_seed55.pt").is_file():
            roots.append(root)
    if not roots:
        extract = Path("/kaggle/working/v55_source_extract")
        extract.mkdir(exist_ok=True)
        for archive in Path("/kaggle/input").rglob("*artifacts.zip"):
            destination = extract / archive.stem
            destination.mkdir(exist_ok=True)
            with zipfile.ZipFile(archive) as zipped:
                zipped.extractall(destination)
        for acceptance in extract.rglob("v55_paper_acceptance.json"):
            root = acceptance.parent
            if (root / "paper_training_handoff/classifier_seed55.pt").is_file():
                roots.append(root)
    unique = [Path(value) for value in sorted(set(map(str, roots)))]
    if len(unique) != 1:
        raise RuntimeError(f"Expected exactly one V5.5 source root, found {unique}")
    return unique[0]

SOURCE = discover_source()
HANDOFF = SOURCE / "paper_training_handoff"
ACCEPTANCE = json.loads((SOURCE / "v55_paper_acceptance.json").read_text())
if not ACCEPTANCE.get("accepted") or not ACCEPTANCE.get("paper_numbers"):
    raise AssertionError("Source is not an accepted V5.5 paper artifact")
if ACCEPTANCE.get("dataset") != DATASET:
    raise AssertionError((ACCEPTANCE.get("dataset"), DATASET))

manifest_path = HANDOFF / "training_handoff_manifest.json"
manifest = json.loads(manifest_path.read_text())
checksum_rows = []
for relative, expected in manifest["files_sha256"].items():
    actual = sha256_file(HANDOFF / relative)
    checksum_rows.append({
        "file": relative, "expected_sha256": expected,
        "actual_sha256": actual, "checksum_match": actual == expected,
    })
checksum_audit = pd.DataFrame(checksum_rows)
if not checksum_audit.checksum_match.all():
    raise AssertionError(checksum_audit.loc[~checksum_audit.checksum_match])
checksum_audit.to_csv(OUTPUT / "01_source_checksum_audit.csv", index=False)

sys.path.insert(0, str(HANDOFF))
import plaintext_cfe_pipeline as pipeline
from plaintext_cfe_pipeline import (
    PreparedData, HEPolynomialMLP, DomainProjector, SearchConfig,
    build_official_dice_explainers, cfe_set_metrics, load_dataset,
    official_dice_generate, representative_query_subset, resolve_input,
)

raw = load_dataset(DATASET, resolve_input(DATASET))
bundle = joblib.load(HANDOFF / "preprocessor_and_split.joblib")
preprocessor, split = bundle["preprocessor"], bundle["split"]
y = raw.frame[raw.target_col].to_numpy(np.int8)
x_train = preprocessor.transform(raw.frame.iloc[split.train_idx]).astype(np.float32)
x_valid = preprocessor.transform(raw.frame.iloc[split.valid_idx]).astype(np.float32)
x_test = preprocessor.transform(raw.frame.iloc[split.test_idx]).astype(np.float32)
prepared = PreparedData(
    raw, split, preprocessor,
    x_train, y[split.train_idx], x_valid, y[split.valid_idx],
    x_test, y[split.test_idx], list(bundle["feature_names"]),
)

checkpoint = torch.load(
    HANDOFF / "classifier_seed55.pt", map_location="cpu", weights_only=False
)
classifier_cfg = checkpoint["classifier_config"]
oracle = HEPolynomialMLP(
    checkpoint["input_dim"], classifier_cfg["h1"], classifier_cfg["h2"],
    classifier_cfg["activation_kind"], classifier_cfg["alpha"],
    classifier_cfg["dropout"],
)
oracle.load_state_dict(checkpoint["state_dict"])
oracle = oracle.eval().to(DEVICE)
projector = DomainProjector(prepared)

all_queries = pd.read_csv(SOURCE / "outer_test_query_manifest_v5.csv")
if RUN_MODE == "paper":
    queries = all_queries.copy()
else:
    queries = representative_query_subset(all_queries, per_direction=PER_DIRECTION)
expected = PER_DIRECTION if RUN_MODE == "paper" else PER_DIRECTION
counts = queries.groupby("direction").size().to_dict()
if counts != {"disease_to_no_disease": expected, "no_disease_to_disease": expected}:
    raise AssertionError(counts)
if queries.query_id.duplicated().any():
    raise AssertionError("Duplicate query_id in evaluation cohort")
queries.to_csv(OUTPUT / "02_exact_query_cohort.csv", index=False)

source_audit = {
    "protocol": PROTOCOL, "dataset": DATASET,
    "source_acceptance_protocol": ACCEPTANCE.get("protocol"),
    "source_accepted": True, "source_paper_numbers": True,
    "source_manifest_sha256": sha256_file(manifest_path),
    "classifier_sha256": sha256_file(HANDOFF / "classifier_seed55.pt"),
    "preprocessor_sha256": sha256_file(HANDOFF / "preprocessor_and_split.joblib"),
    "query_manifest_sha256": sha256_file(SOURCE / "outer_test_query_manifest_v5.csv"),
    "all_handoff_checksums_match": bool(checksum_audit.checksum_match.all()),
    "mlp_retrained": False, "gan_retrained": False,
    "dp_training_or_accountant_changed": False,
}
(OUTPUT / "source_reuse_contract.json").write_text(json.dumps(source_audit, indent=2))
print(pd.DataFrame([source_audit]).to_string(index=False))
print(queries.groupby(["direction", "difficulty_stratum"]).size())


## Official DiCE evaluation

Both methods receive the same factuals, desired labels, immutable/actionable
features, permitted raw ranges, (K=10), margin, classifier and metric code.
DiCE Random uses `sample_size=10,000`; DiCE Genetic uses at most 300 native
iterations. Each factual-method call has a 20-second wall-time cap. Results are
checkpointed after every call.


In [ ]:
dice_explainers, features_to_vary, permitted_range = build_official_dice_explainers(
    prepared, oracle, projector, DEVICE, reference_size=5000
)
metric_config = SearchConfig(population=32, max_rounds=16, k=K, margin=MARGIN)
raw_path = OUTPUT / "03_official_dice_factual_metrics_raw.csv"
rows = []
run_started = time.perf_counter()

for ordinal, record in enumerate(queries.itertuples(index=False), start=1):
    query = prepared.x_test[int(record.local_test_index)]
    desired = int(record.desired_class)
    for method in ["official_dice_random", "official_dice_genetic"]:
        run_seed = (
            8_000_000 + 100_000 * BASELINE_SEED
            + 1000 * int(record.query_id) + sum(map(ord, method))
        )
        cfes, outcome = official_dice_generate(
            method, query, desired, prepared, oracle, projector, DEVICE,
            dice_explainers[method], features_to_vary, permitted_range,
            k=K, seed=run_seed, timeout_seconds=TIMEOUT_SECONDS,
            random_sample_size=RANDOM_SAMPLE_SIZE,
            genetic_maxiterations=GENETIC_MAXITERATIONS,
        )
        metrics = cfe_set_metrics(
            cfes, query, desired, oracle, projector, metric_config, DEVICE,
            outcome["runtime_seconds"], audit=None,
            robustness_seed=run_seed + 7_000_000,
            robustness_repetitions=ROBUSTNESS_REPETITIONS,
        )
        rows.append({
            "query_id": int(record.query_id),
            "local_test_index": int(record.local_test_index),
            "query_class": int(record.query_class),
            "desired_class": desired,
            "direction": record.direction,
            "difficulty_stratum": record.difficulty_stratum,
            "baseline_seed": BASELINE_SEED,
            "run_seed": run_seed,
            "method": method,
            "implementation": "official dice-ml==0.12",
            "K": K, "margin": MARGIN,
            "native_random_sample_size": (
                RANDOM_SAMPLE_SIZE if method == "official_dice_random" else np.nan
            ),
            "native_genetic_maxiterations": (
                GENETIC_MAXITERATIONS if method == "official_dice_genetic" else np.nan
            ),
            **metrics, **outcome,
        })
        pd.DataFrame(rows).to_csv(raw_path, index=False)
    if ordinal == 1 or ordinal % 10 == 0 or ordinal == len(queries):
        elapsed = time.perf_counter() - run_started
        print(f"{ordinal}/{len(queries)} factuals; elapsed={elapsed/60:.1f} min")

raw_results = pd.DataFrame(rows)
print(raw_results.groupby(["method", "direction", "algorithm_status"]).size())


## Unconditional success and success-conditioned quality

Yield, Coverage, Full-K and Robust Yield use every predeclared factual,
including timeouts/no-CF. Proximity, Sparsity, Diversity and Plausibility are
defined only when at least one valid CFE exists, so their effective sample
count is printed explicitly.


In [ ]:
def summarize(group):
    return pd.Series({
        "n_factuals": int(group.query_id.nunique()),
        "n_ok": int((group.algorithm_status == "ok").sum()),
        "n_timeout": int((group.algorithm_status == "timeout").sum()),
        "n_no_cf": int((group.algorithm_status == "no_cf").sum()),
        "n_implementation_error": int((group.algorithm_status == "implementation_error").sum()),
        "n_quality_defined": int(group.proximity.notna().sum()),
        "valid_cfe_yield_at_k": float(group.valid_cfe_yield_at_k.mean()),
        "coverage_at_1": float(group.coverage_at_1.mean()),
        "full_k_success": float(group.full_k_success.mean()),
        "robust_yield_at_k": float(group.robust_completeness_at_k.mean()),
        "robust_full_k_success": float(group.robust_full_k_success.mean()),
        "proximity": float(group.proximity.mean()),
        "sparsity": float(group.sparsity.mean()),
        "diversity": float(group.diversity.mean()),
        "plausibility": float(group.plausibility.mean()),
        "constraint_validity": float(group.constraint_validity.mean()),
        "runtime_seconds_mean": float(group.runtime_seconds.mean()),
        "runtime_seconds_total": float(group.runtime_seconds.sum()),
    })

summary_rows = []
for (method, direction), group in raw_results.groupby(
    ["method", "direction"], sort=True
):
    row = summarize(group).to_dict()
    row.update({"method": method, "direction": direction})
    summary_rows.append(row)
by_direction = pd.DataFrame(summary_rows)
by_direction.to_csv(OUTPUT / "04_official_dice_summary_by_direction.csv", index=False)

# Equal-direction macro: each clinical direction has weight 1/2.
macro_metrics = [
    "valid_cfe_yield_at_k", "coverage_at_1", "full_k_success",
    "robust_yield_at_k", "robust_full_k_success", "proximity", "sparsity",
    "diversity", "plausibility", "constraint_validity", "runtime_seconds_mean",
]
macro = by_direction.groupby("method", as_index=False)[macro_metrics].mean()
macro["aggregation"] = "equal-direction macro (0.5 per direction)"
macro.to_csv(OUTPUT / "05_official_dice_equal_direction_macro.csv", index=False)

outcomes = (
    raw_results.groupby(["method", "direction", "algorithm_status"], as_index=False)
    .size().rename(columns={"size": "n_factuals"})
)
outcomes.to_csv(OUTPUT / "06_official_dice_outcomes.csv", index=False)
display(by_direction)
display(macro)
display(outcomes)


## Linked comparison with the accepted V5.5 table

This table links official DiCE to the accepted paper operating point. The
cohort and metrics are matched. The final columns explicitly prevent a false
claim of equal computational budget: V5.5 custom methods have counted
candidate budgets, while official DiCE has native samples/iterations and
measured wall time.


In [ ]:
main_path = SOURCE / "plaintext_manuscript_tables/03_2a_main_metrics_by_direction.csv"
main = pd.read_csv(main_path)
main = main.loc[main.candidate_budget.eq(OPERATING_BUDGET)].copy()
main_comparison = pd.DataFrame({
    "method": main.method,
    "method_label": main.method_label,
    "direction": main.direction,
    "n_factuals": main.n_factuals.astype(int),
    "valid_cfe_yield_at_k": main.valid_cfe_yield_at_k_mean,
    "coverage_at_1": main.coverage_at_1_mean,
    "full_k_success": main.full_k_success_mean,
    "robust_yield_at_k": main.robust_completeness_at_k_mean,
    "proximity": main.proximity_mean,
    "sparsity": main.sparsity_mean,
    "diversity": main.diversity_mean,
    "plausibility": main.plausibility_mean,
    "constraint_validity": main.constraint_validity_mean,
    "runtime_seconds_mean": main.generation_time_to_k_or_cap_seconds_mean,
    "access_stratum": "custom counted-candidate implementation",
    "candidate_budget": main.candidate_budget,
    "native_control": "counted candidate evaluations",
    "same_cohort_constraints_metrics": True,
    "equal_compute_budget_claimed": False,
})
official_comparison = by_direction.copy()
official_comparison["method_label"] = official_comparison.method.map({
    "official_dice_random": "Official DiCE Random",
    "official_dice_genetic": "Official DiCE Genetic",
})
official_comparison["access_stratum"] = "official dice-ml native implementation"
official_comparison["candidate_budget"] = np.nan
official_comparison["native_control"] = official_comparison.method.map({
    "official_dice_random": f"sample_size={RANDOM_SAMPLE_SIZE}",
    "official_dice_genetic": f"maxiterations={GENETIC_MAXITERATIONS}",
})
official_comparison["same_cohort_constraints_metrics"] = True
official_comparison["equal_compute_budget_claimed"] = False

columns = [
    "method", "method_label", "direction", "n_factuals",
    "valid_cfe_yield_at_k", "coverage_at_1", "full_k_success",
    "robust_yield_at_k", "proximity", "sparsity", "diversity",
    "plausibility", "constraint_validity", "runtime_seconds_mean",
    "access_stratum", "candidate_budget", "native_control",
    "same_cohort_constraints_metrics", "equal_compute_budget_claimed",
]
linked = pd.concat([
    main_comparison[columns], official_comparison[columns]
], ignore_index=True)
linked.to_csv(OUTPUT / "07_linked_fair_comparison_by_direction.csv", index=False)

fairness = pd.DataFrame([
    {"item": "outer-test factual cohort", "matched": True,
      "detail": f"same frozen query IDs; {PER_DIRECTION} per direction"},
    {"item": "desired-label directions", "matched": True,
      "detail": "disease->no disease and no disease->disease reported separately"},
    {"item": "classifier/preprocessor", "matched": True,
      "detail": "checksum-locked V5.5 checkpoint and fitted transform"},
    {"item": "K / margin / constraints", "matched": True,
      "detail": "K=10, logit margin=0.10, same DomainProjector"},
    {"item": "failure denominator", "matched": True,
      "detail": "timeout and no-CF remain zero-yield factuals"},
    {"item": "random seed count", "matched": True,
      "detail": "one predeclared baseline seed (11), matching one-seed V5.5 scope"},
    {"item": "candidate/query budget", "matched": False,
      "detail": "official DiCE native controls differ; runtime and controls reported"},
    {"item": "model training", "matched": True,
      "detail": "none repeated; accepted frozen artifacts reused"},
])
fairness.to_csv(OUTPUT / "08_fairness_audit.csv", index=False)
display(linked)
display(fairness)


In [ ]:
expected_rows = len(queries) * 2
pair_counts = raw_results.groupby(["query_id", "method"]).size()
checks = {
    "source_accepted": bool(ACCEPTANCE.get("accepted")),
    "source_paper_numbers": bool(ACCEPTANCE.get("paper_numbers")),
    "all_source_checksums_match": bool(checksum_audit.checksum_match.all()),
    "exact_direction_counts": bool(
        queries.groupby("direction").size().eq(PER_DIRECTION).all()
    ),
    "exact_result_rows": len(raw_results) == expected_rows,
    "one_row_per_query_method": bool(pair_counts.eq(1).all()),
    "both_official_methods_present": set(raw_results.method) == {
        "official_dice_random", "official_dice_genetic"
    },
    "no_implementation_errors": not bool(
        raw_results.algorithm_status.eq("implementation_error").any()
    ),
    "timeouts_retained": True,
    "no_cf_retained": True,
    "no_mlp_retraining": True,
    "no_gan_retraining": True,
    "no_dp_accountant_change": True,
    "equal_compute_budget_not_claimed": True,
}
acceptance = {
    "protocol": PROTOCOL, "dataset": DATASET, "run_mode": RUN_MODE,
    "accepted": bool(all(checks.values())), "checks": checks,
    "n_queries": int(len(queries)), "queries_per_direction": PER_DIRECTION,
    "n_result_rows": int(len(raw_results)), "K": K, "margin": MARGIN,
    "wall_time_seconds": float(time.perf_counter() - run_started),
}
(OUTPUT / "official_dice_extension_acceptance.json").write_text(
    json.dumps(acceptance, indent=2)
)

archive = shutil.make_archive(str(OUTPUT) + "_artifacts", "zip", OUTPUT)
print(json.dumps(acceptance, indent=2))
print("ARTIFACT_ZIP", archive)
if not acceptance["accepted"]:
    print("WARNING: acceptance checks failed; preserve the ZIP for diagnosis")
